上面完成的任务是：给定一个topic = "I support Trump" 和一段文本，判断这段文本是否能推断出该topic，默认输出的entailment 的概率>0.5 即认为能推断出，将DeBERTa‑v2‑xxlarge‑MNLI 模型简述为ML,上面这个任务可以简化为谓词 ML(topic="I support Trump" ，body=".........").entailment.probability > 0.5 , 由于模型太大，对于20000条文本数据推理代价太大，我考虑训练一个小的proxy模型，默认为二元谓词模型输入两段文本topic 和body，直接给出满足的概率，我选择什么样的小NLP模型训练合适， 已知我有4000条通过DeBERTa‑v2‑xxlarge‑MNLI 模型输出的满足谓词的概率
已经将DeBERTa‑v2‑xxlarge‑MNLI 对topic符合（entailment）body的概率存储在post_LLM_cleaned_6.csv文件中，列名为ML1_oracle2_probability，文本列名为body，从post_LLM_cleaned_6 文件中随机抽样20%的数据用于训练微调distilbert-base-uncased

In [ ]:
import time

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# 1) 选择模型
model_name = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2) 自定义 Dataset（同你之前的 soft-label NLI 脚本）
class ProxyDataset(Dataset):
    def __init__(self, df):
        self.texts = df['body'].tolist()
        self.softs = df['ML1_oracle2_probability'].tolist()

    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx],
                        f"This text is about I support Trump.",
                        truncation=True, padding='max_length', max_length=256,
                        return_tensors='pt')
        soft = torch.tensor([1-self.softs[idx], self.softs[idx]])
        return {**{k:v.squeeze(0) for k,v in enc.items()}, 'soft_label': soft}

# 3) 蒸馏 Trainer
class DistillTrainer(Trainer):
    def __init__(self, T=2.0, **kwargs):
        super().__init__(**kwargs)
        self.T = T
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        soft = inputs.pop("soft_label").to(model.device)
        logits = model(**inputs).logits
        log_p  = F.log_softmax(logits/self.T, dim=-1)
        loss   = F.kl_div(log_p, soft, reduction='batchmean') * (self.T*self.T)
        return (loss, logits) if return_outputs else loss

# 4) 训练
import pandas as pd
df = pd.read_csv("your_4000_samples.csv")
ds = ProxyDataset(df)
args = TrainingArguments("distil-proxy", per_device_train_batch_size=16,
                         num_train_epochs=5, learning_rate=3e-5)
trainer = DistillTrainer(model=model, tokenizer=tokenizer,
                         args=args, train_dataset=ds, T=2.0)
trainer.train()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch, torch.nn.functional as F
from torch.utils.data import Dataset
import pandas as pd
import os

# 禁用 tokenizers 并行警告
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1) 选择模型
model_name = "/home/wangshuo/resource/AIModels/NLP/base-uncased/distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2) 自定义 Dataset（软标签蒸馏用）
class ProxyDataset(Dataset):
    def __init__(self, df):
        self.texts = df['body'].tolist()
        self.softs = df['ML1_oracle2_probability'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            "This text is about I support Trump.",
            truncation=True,
            padding='max_length',
            max_length=256,
            return_tensors='pt'
        )
        # soft label: [p(not), p(entail)]
        soft = torch.tensor([1.0 - self.softs[idx], self.softs[idx]], dtype=torch.float)
        return {
            'input_ids':     enc['input_ids'].squeeze(0),
            'attention_mask':enc['attention_mask'].squeeze(0),
            'soft_label':    soft
        }

# 3) 蒸馏 Trainer
class DistillTrainer(Trainer):
    def __init__(self, T=2.0, **kwargs):
        super().__init__(**kwargs)
        self.T = T

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 拿出软标签
        soft = inputs.pop("soft_label").to(model.device)
        # 前向
        logits = model(**inputs).logits           # [batch, 2]
        # 温度缩放 + KL 散度
        log_p  = F.log_softmax(logits / self.T, dim=-1)
        loss   = F.kl_div(log_p, soft, reduction='batchmean') * (self.T * self.T)
        return (loss, logits) if return_outputs else loss

# 4) 读取原始推理结果，抽样 20%
data_dir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
input_csv = os.path.join(data_dir, 'post_LLM_cleaned_6.csv')
df        = pd.read_csv(input_csv)
df_sample = df.sample(frac=0.2, random_state=42).reset_index(drop=True)

# 构造 Dataset
train_ds = ProxyDataset(df_sample)

# 5) 训练参数
output_dir = '/home/wangshuo/resource/AIModels/Finetune/distil-proxy'
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=500,
    evaluation_strategy="no",
    disable_tqdm=False,
    report_to="none"
)

# 6) 启动蒸馏微调
trainer = DistillTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    T=2.0
)
trainer.train()

# 7) 保存微调后模型
model.save_pretrained(os.path.join(output_dir, 'final'))
tokenizer.save_pretrained(os.path.join(output_dir, 'final'))


一般训练一个代理模型，比如判断谓词为满足概率的小型机器学习模型需要训练多少个batch， 以NLP问题为例，假设我oracle模型是deberta-v2-xxlarge-mnli 模型，给定两段文本post和topic，输入到oracle模型中oracle判断post能否推出topic， 代理模型是不是一个小tranformer模型，提取两个文本特征，再加一个分类头，有没有更好的代理模型的方式，代理模型以oracle的概率输出为软标签，训练

#### vector + MLP
加载训练好的 MLP 代理模型，并对 post_new.csv 中的 body 文本批量推理，将预测出的概率写入新的列并保存到 CSV。

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import joblib
from tqdm.auto import tqdm

# ─── 1. 参数与路径 ────────────────────────────────────────
DATA_DIR     = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
INPUT_CSV    = os.path.join(DATA_DIR, 'post_LLM_cleaned_6.csv')
OUTPUT_CSV   = os.path.join(DATA_DIR, 'post_LLM_cleaned_6.csv')

MODEL_NAME   = '/home/wangshuo/resource/AIModels/NLP/base-uncased/distilbert-base-uncased'
PROXY_DIR    = '/home/wangshuo/resource/AIModels/Finetune/prosy_mlp/'
MLP_PATH     = os.path.join(PROXY_DIR, 'mlp_proxy_epoch200.pkl')   # 你保存的 MLP 文件
TOPIC        = "I support Trump"
BATCH_SIZE   = 64
MAX_LENGTH   = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── 2. 加载 Encoder 与 Tokenizer ─────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder   = AutoModel.from_pretrained(MODEL_NAME).to(device)
encoder.half()
encoder.eval()

# ─── 3. 加载 MLP 代理模型 ─────────────────────────────────
mlp = joblib.load(MLP_PATH)

# ─── 4. 定义文本编码与特征生成 ────────────────────────────
def embed_texts(texts):
    all_vecs = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        enc   = tokenizer(batch, padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors='pt').to(device)
        with torch.no_grad():
            out = encoder(**enc).last_hidden_state  # [B, L, H]
        vecs = out.mean(dim=1).cpu().numpy()       # mean-pooling
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

def make_features(bodies):
    # 1) 句向量: topic vs body
    emb_topic = embed_texts([TOPIC] * len(bodies))
    emb_body  = embed_texts(bodies)
    # 2) 拼接特征
    return np.hstack([
        emb_topic,
        emb_body,
        emb_topic * emb_body,
        np.abs(emb_topic - emb_body)
    ])

# ─── 5. 批量推理 ─────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
bodies = df['body'].fillna("").astype(str).tolist()

proxy_probs = []
for i in tqdm(range(0, len(bodies), BATCH_SIZE), desc="Proxy Inference"):
    batch_texts = bodies[i:i+BATCH_SIZE]
    feats       = make_features(batch_texts)
    preds       = mlp.predict(feats)          # 输出概率
    proxy_probs.extend(preds)

# ─── 6. 写回并保存 ────────────────────────────────────────
df['ML1_proxy8_probability'] = proxy_probs
df.to_csv(OUTPUT_CSV, index=False)
print(f"推理完成，结果已保存到：{OUTPUT_CSV}")


#### vector + SVM

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import joblib
from tqdm.auto import tqdm

# ─── 1. 参数与路径 ────────────────────────────────────────
DATA_DIR   = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
INPUT_CSV  = os.path.join(DATA_DIR, 'post_LLM_cleaned_6.csv')
OUTPUT_CSV = os.path.join(DATA_DIR, 'post_LLM_cleaned_6.csv')

DISTIL_MODEL = '/home/wangshuo/resource/AIModels/NLP/base-uncased/distilbert-base-uncased'
SVM_MODEL    = '/home/wangshuo/resource/AIModels/Finetune/proxy_svm/proxy_svm.pkl'
TOPIC        = "I support Trump"
BATCH_SIZE   = 64
MAX_LENGTH   = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── 2. 加载 DistilBERT 编码器 ────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(DISTIL_MODEL)
encoder   = AutoModel.from_pretrained(DISTIL_MODEL).to(device)
encoder.eval()

# ─── 3. 加载 SVM 模型 ─────────────────────────────────────
svm = joblib.load(SVM_MODEL)

# ─── 4. 定义句向量 & 特征函数 ─────────────────────────────
def embed_texts(texts):
    all_vecs = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        enc   = tokenizer(batch,
                          padding=True,
                          truncation=True,
                          max_length=MAX_LENGTH,
                          return_tensors='pt').to(device)
        with torch.no_grad():
            last_hidden = encoder(**enc).last_hidden_state  # [B,L,H]
        vecs = last_hidden.mean(dim=1).cpu().numpy()      # mean-pooling
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

def make_features(bodies):
    # topic 向量
    emb_topic = embed_texts([TOPIC] * len(bodies))
    # body 向量
    emb_body  = embed_texts(bodies)
    # 拼接 [t, b, t*b, |t-b|]
    return np.hstack([
        emb_topic,
        emb_body,
        emb_topic * emb_body,
        np.abs(emb_topic - emb_body)
    ])

# ─── 5. 批量推理 ─────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
texts = df['body'].fillna("").astype(str).tolist()

proxy_probs = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="SVM Inference"):
    batch = texts[i:i+BATCH_SIZE]
    feats = make_features(batch)
    # SVM 的 predict_proba 返回 shape [N, 2], 第二列为正例（entailment）的概率
    probs = svm.predict_proba(feats)[:, 1]
    proxy_probs.extend(probs)

# ─── 6. 写回 & 保存 ────────────────────────────────────────
df['ML1_proxy9_probability'] = proxy_probs
df.to_csv(OUTPUT_CSV, index=False)
print(f"推理完成，结果已保存到：{OUTPUT_CSV}")


#### vector + linear

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import joblib
from tqdm.auto import tqdm

# ─── 1. 参数与路径 ────────────────────────────────────────
DATA_DIR        = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5'
INPUT_CSV       = os.path.join(DATA_DIR, 'post_LLM_cleaned_6.csv')
OUTPUT_CSV      = os.path.join(DATA_DIR, 'post_LLM_cleaned_6.csv')

DISTIL_MODEL    = '/home/wangshuo/resource/AIModels/NLP/base-uncased/distilbert-base-uncased'
LINEAR_MODEL    = '/home/wangshuo/resource/AIModels/Finetune/proxy_linear/proxy_linear.pkl'
TOPIC           = "I support Trump"
BATCH_SIZE      = 64
MAX_LENGTH      = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── 2. 加载 DistilBERT 编码器 ────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(DISTIL_MODEL)
encoder   = AutoModel.from_pretrained(DISTIL_MODEL).to(device)
encoder.eval()

# ─── 3. 加载 LinearRegression 模型 ───────────────────────
lr = joblib.load(LINEAR_MODEL)

# ─── 4. 定义句向量 & 特征函数 ─────────────────────────────
def embed_texts(texts):
    all_vecs = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        enc   = tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=MAX_LENGTH,
                    return_tensors='pt'
               ).to(device)
        with torch.no_grad():
            last_hidden = encoder(**enc).last_hidden_state  # [B, L, H]
        vecs = last_hidden.mean(dim=1).cpu().numpy()      # mean-pooling
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

def make_features(bodies):
    emb_topic = embed_texts([TOPIC] * len(bodies))
    emb_body  = embed_texts(bodies)
    return np.hstack([
        emb_topic,
        emb_body,
        emb_topic * emb_body,
        np.abs(emb_topic - emb_body)
    ])

# ─── 5. 批量推理 ─────────────────────────────────────────
df    = pd.read_csv(INPUT_CSV)
texts = df['body'].fillna("").astype(str).tolist()

proxy_probs = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="LinearProxy Inference"):
    batch_texts = texts[i:i+BATCH_SIZE]
    feats       = make_features(batch_texts)
    preds       = lr.predict(feats)           # 线性回归直接输出概率估计
    proxy_probs.extend(preds)

# ─── 6. 写回 & 保存 ────────────────────────────────────────
df['ML1_proxy_linear_probability'] = proxy_probs
df.to_csv(OUTPUT_CSV, index=False)
print(f"推理完成，结果已保存到：{OUTPUT_CSV}")


### 二、 

In [ ]:
import pandas as pd
import numpy as np
from gensim import downloader
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# — 1. 读数据 & 抽样 —
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
df_full = pd.read_csv(datadir + 'comment_test.csv')
# 随机抽取 10% 的行
df = df_full.sample(frac=0.1, random_state=42).reset_index(drop=True)

texts = df['body'].astype(str)
y     = df['ML2_oracle1_probability'].values

# — 2. 加载小维度推文向量 —
w2v = downloader.load("glove-twitter-50")  # 可选 25/50/100/200 维

# — 3. 文本向量化（平均词向量） —
def embed(text):
    vecs = [w2v[w] for w in text.split() if w in w2v]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)

X = np.vstack(texts.apply(embed).values)

# — 4. 拆分、训练、评估 —
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

svr = SVR(kernel='linear', C=1.0, epsilon=0.01)
svr.fit(X_tr, y_tr)

y_pred = svr.predict(X_te)
rmse = mean_squared_error(y_te, y_pred, squared=False)
print(f"使用 10% 样本 ({len(df)}/{len(df_full)}) 训练，测试集 RMSE: {rmse:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from gensim import downloader
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
from tqdm.auto import tqdm  # 进度条

# — 1. 读数据 & 抽样 —
datadir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
proxydir = '/home/wangshuo/resource/AIModels/Finetune/TE/proxy_mlp/'
df_full  = pd.read_csv(datadir + 'comment_test.csv')
df       = df_full.sample(frac=0.1, random_state=42).reset_index(drop=True)

texts = df['body'].astype(str)
y     = df['ML2_oracle1_probability'].values

# — 2. 加载推文向量 —
print('加载词向量中...')
w2v = downloader.load("glove-twitter-25")

print('加载词向量完成，维度:', w2v.vector_size)
# — 3. 文本向量化（平均词向量），带进度条 —
def embed(text):
    vecs = [w2v[w] for w in text.split() if w in w2v]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)

# 用 for 循环和 tqdm 构造 X
embeddings = []
for text in tqdm(texts, desc="Embedding texts", unit="doc"):
    embeddings.append(embed(text))
X = np.vstack(embeddings)

# — 4. 拆分、训练 MLP 回归模型 —
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlp = MLPRegressor(
    hidden_layer_sizes=(100, 100),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate='adaptive',
    max_iter=200,
    random_state=42,
    verbose=True
)
mlp.fit(X_tr, y_tr)

# — 5. 评估 —
y_pred = mlp.predict(X_te)
rmse = mean_squared_error(y_te, y_pred, squared=False)
print(f"Test RMSE (MLP): {rmse:.4f}")

# — 6. 保存模型 —
model_path = proxydir + 'mlp_glove-twitter-50.joblib'
joblib.dump(mlp, model_path)
print(f"MLP 模型已保存至: {model_path}")


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
from tqdm.auto import tqdm

# — 0. 设备配置 — 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# — 1. 读数据 & 抽样 —
datadir  = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
proxydir = '/home/wangshuo/resource/AIModels/Finetune/TE/proxy_mlp/'
df_full  = pd.read_csv(datadir + 'comment_test.csv')
df       = df_full.sample(frac=0.2, random_state=42).reset_index(drop=True)

texts = df['body'].astype(str).tolist()
y     = df['ML2_oracle1_probability'].values

# — 2. 加载 TinyBERT 模型 & Tokenizer —
model_name = "/home/wangshuo/resource/AIModels/NLP/base-uncased/distilbert-base-uncased"  # TinyBERT 4-layer, 312-hidden
print("Loading TinyBERT model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModel.from_pretrained(model_name).to(device)
model.eval()
hidden_size = model.config.hidden_size
print(f"TinyBERT hidden size: {hidden_size}")

# — 3. 文本向量化函数 — 
#    取所有 token 的平均向量作为句子表示
@torch.no_grad()
def embed_tinybert(text: str) -> np.ndarray:
    encoded = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors="pt"
    ).to(device)
    outputs = model(**encoded)
    # last_hidden_state: [batch=1, seq_len, hidden_size]
    last_hid = outputs.last_hidden_state.squeeze(0)
    # 平均所有 token
    vec = last_hid.mean(dim=0).cpu().numpy()
    return vec

# — 4. 构造特征矩阵 X（带进度条） —
print("Embedding texts with TinyBERT...")
embeddings = []
for txt in tqdm(texts, desc="TinyBERT embedding", unit="doc"):
    embeddings.append(embed_tinybert(txt))
X = np.vstack(embeddings)  # shape = [n_samples, hidden_size]

# — 5. 拆分训练/测试集 —
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# — 6. 训练 MLP 回归模型 —
mlp = MLPRegressor(
    hidden_layer_sizes=(100, 100),
    activation='relu',
    solver='adam',
    alpha=1e-4,            # L2 正则化强度
    learning_rate='adaptive',
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
    verbose=True
)
print("Training MLP regressor...")
mlp.fit(X_tr, y_tr)

# — 7. 在测试集上评估 —
y_pred = mlp.predict(X_te)
rmse = mean_squared_error(y_te, y_pred, squared=False)
print(f"Test RMSE (MLP w/ TinyBERT): {rmse:.4f}")

# — 8. 保存模型 & 设备信息 —
model_path  = proxydir + 'mlp_tinybert_proxy.joblib'
tokenizer.save_pretrained(proxydir + 'tinybert_tokenizer')
model.save_pretrained(proxydir + 'tinybert_model')
joblib.dump(mlp, model_path)
print(f"Saved MLP proxy model to: {model_path}")


In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from scipy.special import softmax
from tqdm import tqdm

# —— 1. 模型路径或名称 ——
# MODEL_NAME = "/home/wangshuo/resource/AIModels/Finetune/TE/distill_roberta-base-SST-2"
MODEL_NAME = "/home/wangshuo/resource/AIModels/Finetune/TE/distill_bert-mini-finetuned-sst2"

# —— 2. 手动加载 tokenizer 和 model ——
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device    = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
# model.half()  # 使用半精度加速
model.to(device).eval()

# —— 3. 从 config 中读取 id2label ——
#     DistilBERT 的 config 中自带 {"0":"NEGATIVE","1":"POSITIVE"}
id2label = model.config.id2label

# —— 4. （可选）预处理函数 ——
def preprocess(text: str) -> str:
    return text.strip()

# —— 5. 读取 CSV ——
datadir  = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5"
in_csv   = f"{datadir}/comment_test.csv"
out_csv  = f"{datadir}/comment_test.csv"
df       = pd.read_csv(in_csv)
texts    = df['body'].fillna("").astype(str).tolist()

# —— 6. 批量推理函数 ——
def infer_batch(batch_texts):
    # 分词 + 编码
    inputs = tokenizer(
        [preprocess(t) for t in batch_texts],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    # 推理
    with torch.no_grad():
        logits = model(**inputs).logits.cpu().numpy()

    # softmax → 概率
    probs = softmax(logits, axis=1)

    # 取最优类别及其概率
    pred_ids    = probs.argmax(axis=1)
    pred_scores = probs.max(axis=1)
    pred_labels = [id2label[i] for i in pred_ids]

    return pred_labels, pred_scores

# —— 7. 分批处理所有文本 ——
batch_size = 1024
all_labels = []
all_scores = []

for i in tqdm(range(0, len(texts), batch_size), desc="DistilBERT 推理"):
    batch = texts[i : i + batch_size]
    lbs, scs = infer_batch(batch)
    all_labels.extend(lbs)
    all_scores.extend(scs)

# —— 8. 写回 CSV ——
df['ML2_proxy7_label'] = all_labels
df['ML2_proxy7_probability'] = all_scores
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"tiny-BERT 情感分析结果已保存到：{out_csv}")


DistilBERT 推理: 100%|██████████| 25/25 [00:15<00:00,  1.58it/s]


tiny-BERT 情感分析结果已保存到：/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment_test.csv


下面是我使用两个模型进行情感分析二分类的代码， 由于bert-large-uncased-sst2 推理速度太慢，在TinyBERT-4L-312D-SST-2 基础上微调或者说蒸馏bert-large-uncased-sst2 学习的概率分布，现已经有bert-large-uncased-sst2 在comment_test上的输出，学习，给出高质量的代码

In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

# ——————————————
# 0. 全局超参与路径
# ——————————————
model_name = ''
MODEL_STUDENT = "/home/wangshuo/resource/AIModels/NLP/NLI/bert-mini-finetuned-mnli"
DATA_CSV = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv"
OUTPUT_DIR = "/home/wangshuo/resource/AIModels/Finetune/NLI/distilled_bert-mini-finetuned-mnli/"
BATCH_SIZE = 16
EPOCHS = 1
LR = 3e-5
TEMPERATURE = 2.0
ALPHA = 0.6  # soft vs hard loss 权重
MAX_LEN = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ——————————————
# 1. Dataset：读取 body, 构造 hypothesis, 加载教师三分类软标签
# ——————————————
class NLIDistillDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 512):
        self.premises = df["body"].fillna("").astype(str).tolist()
        # 固定 hypothesis
        self.hypotheses = ["This text is about I support Trump." for _ in self.premises]
        # 教师在 CSV 中已存的三分类概率列
        probs = df[[
            "ML1_oracle1_contra",
            "ML1_oracle1_neutral",
            "ML1_oracle1_entail"
        ]].values.astype(float)
        self.teacher_probs = torch.tensor(probs, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.premises[idx],
            self.hypotheses[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["teacher_probs"] = self.teacher_probs[idx]
        return item


# ——————————————
# 2. 加载学生模型 & tokenizer
# ——————————————
tokenizer_s = AutoTokenizer.from_pretrained(MODEL_STUDENT)
student = AutoModelForSequenceClassification.from_pretrained(
    MODEL_STUDENT,
    num_labels=3,  # 三分类头
    ignore_mismatched_sizes=True  # in case original head !=3
).to(device)
student.train()

# ——————————————
# 3. 构建 DataLoader
# ——————————————
df_full = pd.read_csv(DATA_CSV)
# 可选：抽样加速实验
df = df_full.sample(frac=0.2, random_state=42).reset_index(drop=True)
dataset = NLIDistillDataset(df, tokenizer_s, max_len=MAX_LEN)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# ——————————————
# 4. 损失 / 优化器 / Scheduler
# ——————————————
ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction="batchmean")
optimizer = AdamW(student.parameters(), lr=LR)
total_steps = len(loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# ——————————————
# 5. 训练循环：混合软硬标签
# ——————————————
for epoch in range(EPOCHS):
    pbar = tqdm(loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    epoch_loss = 0.0

    for batch in pbar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        teacher_p = batch["teacher_probs"].to(device)  # [B,3]

        # 学生前向
        outputs = student(input_ids=input_ids, attention_mask=attention_mask)
        logits_s = outputs.logits  # [B,3]

        # —— 1) 硬标签 CE Loss ——
        hard_labels = teacher_p.argmax(dim=1)  # [B]
        loss_ce = ce_loss(logits_s, hard_labels)

        # —— 2) 软标签 KD Loss ——
        log_p_s = nn.functional.log_softmax(logits_s / TEMPERATURE, dim=1)
        p_t = nn.functional.softmax(teacher_p / TEMPERATURE, dim=1)
        loss_kd = kl_loss(log_p_s, p_t) * (TEMPERATURE ** 2)

        # —— 3) 混合总损失 ——
        loss = ALPHA * loss_kd + (1 - ALPHA) * loss_ce
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix(avg_loss=epoch_loss / (pbar.n + 1))

    print(f"→ Epoch {epoch + 1} avg loss: {epoch_loss / len(loader):.4f}")

# ——————————————
# 6. 保存微调好的学生模型
# ——————————————
student.save_pretrained(OUTPUT_DIR)
tokenizer_s.save_pretrained(OUTPUT_DIR)
print(f"✅ Distilled model saved to {OUTPUT_DIR}")


/home/wangshuo/software/anaconda3/envs/iogs/lib/python3.8/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/1:   0%|          | 0/308 [00:00<?, ?it/s]

→ Epoch 1 avg loss: 0.4134
[2025-07-22 15:50:28,016] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/wangshuo/software/anaconda3/envs/iogs/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


MissingCUDAException: CUDA_HOME does not exist, unable to compile CUDA op(s)

base1 -fintune

In [1]:
import os
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# —————— 配置 ——————
model_name = 'deberta-v3-large-binary-epoch1'
MODEL_DIR = f"/home/wangshuo/resource/AIModels/Finetune/NLI/base/{model_name}"  # 你的模型与 tokenizer 保存目录
INPUT_CSV = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv"  # 待推理的文件
OUTPUT_CSV = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv"  # 覆盖写回
BATCH_SIZE = 32
MAX_LEN = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# —————— 加载模型 & tokenizer ——————
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.half()
model.eval()

# —————— 读取数据 ——————
df = pd.read_csv(INPUT_CSV)
posts = df['body'].fillna("").astype(str).tolist()


# —————— 推理函数 ——————
def infer_entail_over_contra(posts_batch):
    enc = tokenizer(
        posts_batch,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits  # [B,2]
    # 只取 contra(0) 和 entail(1) 两列，softmax 归一化
    two_logits = logits[:, [0, 1]]
    probs = two_logits.softmax(dim=1)
    # 返回 entail 概率
    return probs[:, 1].cpu().numpy()


# —————— 批量推理 ——————
proxy_probs = []
batch_times = []
for i in tqdm(range(0, len(posts), BATCH_SIZE), desc="Inferencing"):
    batch = posts[i: i + BATCH_SIZE]
    t0 = time.perf_counter()
    proxy_probs.extend(infer_entail_over_contra(batch))
    t1 = time.perf_counter()
    batch_times.append(t1 - t0)

min_time = min(batch_times)
max_time = max(batch_times)
avg_time = sum(batch_times) / len(batch_times)
# 吞吐量 = batch_size / 时间
max_throughput = 1 / min_time  # 最快时的最大吞吐
min_throughput = 1 / max_time  # 最慢时的最小吞吐
avg_throughput = 1 / avg_time  # 平均吞吐

print(f"每个 batch 推理耗时 (s)：最短={min_time:.4f}, 最长={max_time:.4f}, 平均={avg_time:.4f}")
print(f"吞吐量 (batch/s)：最大={max_throughput:.2f}, 最小={min_throughput:.2f}, 平均={avg_throughput:.2f}")
# —————— 写回结果 ——————
df['ML1_proxy4bb_probability'] = proxy_probs
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ 推理完成，结果保存到 {OUTPUT_CSV}")


Inferencing:   0%|          | 0/769 [00:00<?, ?it/s]

每个 batch 推理耗时 (s)：最短=0.0632, 最长=0.7502, 平均=0.1402
吞吐量 (batch/s)：最大=15.83, 最小=1.33, 平均=7.13
✅ 推理完成，结果保存到 /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv


base2-finetune

In [4]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# —————— 配置 ——————
MODEL_DIR   = "/home/wangshuo/resource/AIModels/Finetune/NLI/deberta-v3-base-p"        # 你的模型与 tokenizer 保存目录
INPUT_CSV   = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv"         # 待推理的文件
OUTPUT_CSV  = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv"         # 覆盖写回
BATCH_SIZE   = 32
MAX_LEN      = 256
TOPIC        = "I support Trump"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# —————— 加载 ——————
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.half()
model.eval()

# —————— 读取数据 ——————
df    = pd.read_csv(INPUT_CSV)
posts = df['body'].fillna("").astype(str).tolist()

# —————— 推理函数 ——————
def infer_entail_over_contra(posts_batch):
    # 构造对应的 hypothesis 列表
    hypotheses = [f"This text is about {TOPIC}." for _ in posts_batch]
    enc = tokenizer(
        posts_batch,
        hypotheses,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits  # [B,3]
    # 只取 contra(0) 和 entail(2)
    two_logits = logits[:, [0, 2]]
    probs      = two_logits.softmax(dim=1)
    return probs[:, 1].cpu().numpy()  # entail 概率

# —————— 批量推理 ——————
all_probs = []
for i in tqdm(range(0, len(posts), BATCH_SIZE), desc="Inferencing"):
    batch = posts[i:i + BATCH_SIZE]
    all_probs.extend(infer_entail_over_contra(batch))

# —————— 写回结果 ——————
df['ML1_proxy7_probability'] = all_probs
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ 推理完成，结果已保存到 {OUTPUT_CSV}")


Inferencing:   0%|          | 0/769 [00:00<?, ?it/s]

✅ 推理完成，结果已保存到 /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/post_origin.csv


In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

teacher_model_name = 'roberta-large-sst2'
student_model_name = 'TinyBERT-4L-312D-SST-2'  # TinyBERT-4L-312D, 2-class head
dataset_name = 'comment_test'
oralce_label = 'oracle2'

# — 0. 参数设置 —
BATCH_SIZE = 64
EPOCHS = 15
LR = 5e-5
TEMPERATURE = 2.0
ALPHA = 0.7  # 蒸馏损失 vs. 交叉熵损失的权重
MODEL_TEACHER = f"/home/wangshuo/resource/AIModels/NLP/TE/{teacher_model_name}"
MODEL_STUDENT = f"/home/wangshuo/resource/AIModels/NLP/TE/{student_model_name}"  # TinyBERT-4L-312D, 2-class head
CSV_PATH = f"/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/{dataset_name}.csv"
OUTPUT_DIR = f"/home/wangshuo/resource/AIModels/Finetune/TE/distil/distill_{oralce_label}_{student_model_name}{EPOCHS}/"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 打开日志文件，写表头
log_path = os.path.join(OUTPUT_DIR, "log.txt")
with open(log_path, "w") as log_f:
    log_f.write("Epoch\tTrainLoss\tValLoss\tValAcc\n")

# — 1. 数据集定义 — 包含文本 和 “软标签” teacher_probs
class SST2DistillDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df["body"].fillna("").astype(str).tolist()
        # teacher 只给了正类概率 p; 构造 [neg_prob, pos_prob]
        p_pos = df["ML2_oracle2_probability"].values.astype(float)
        p_neg = 1.0 - p_pos
        self.teacher_probs = torch.tensor(
            list(zip(p_neg, p_pos)), dtype=torch.float32
        )
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["teacher_probs"] = self.teacher_probs[idx]
        return item


# — 2. 加载 tokenizer 和模型 —
tokenizer_t = AutoTokenizer.from_pretrained(MODEL_TEACHER)
teacher = AutoModelForSequenceClassification.from_pretrained(
    MODEL_TEACHER
).to(device)
teacher.eval()  # 只取过来的软标签，本例不在训练中调用 teacher.forward

tokenizer_s = AutoTokenizer.from_pretrained(MODEL_STUDENT)
student = AutoModelForSequenceClassification.from_pretrained(
    MODEL_STUDENT, num_labels=2
).to(device)

# — 3. 读取并切分数据 —
# — 3. 读取、抽样并切分数据 —
df_full = pd.read_csv(CSV_PATH)
# 随机抽取 20% 的样本
df = df_full.sample(frac=0.2, random_state=42).reset_index(drop=True)
dataset = SST2DistillDataset(df, tokenizer_s)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# — 4. 损失函数 & 优化器 & 学习率调度器 —
# 交叉熵（硬标签从 teacher_probs 二值化而来）
ce_loss = nn.CrossEntropyLoss()
# KL 散度用于软标签
kl_loss = nn.KLDivLoss(reduction="batchmean")
optimizer = AdamW(student.parameters(), lr=LR)
total_steps = len(loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

# — 5. 训练循环 —
student.train()
for epoch in range(EPOCHS):
    loop = tqdm(loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    running_loss = 0.0
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        teacher_p = batch["teacher_probs"].to(device)  # shape [B,2]

        # Student 前向
        outputs = student(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        logits_s = outputs.logits  # shape [B,2]

        # 1) 硬标签：从 teacher_p 选 max
        hard_labels = teacher_p.argmax(dim=1)

        # 2) 交叉熵 Loss
        loss_ce = ce_loss(logits_s, hard_labels)

        # 3) 软标签 Distillation Loss (温度缩放后 KLDiv)
        #    log_softmax(student_logits / T)
        log_p_s = nn.functional.log_softmax(logits_s / TEMPERATURE, dim=1)
        p_t = nn.functional.softmax(teacher_p / TEMPERATURE, dim=1)
        loss_kd = kl_loss(log_p_s, p_t) * (TEMPERATURE ** 2)

        # 4) 总损失
        loss = ALPHA * loss_kd + (1 - ALPHA) * loss_ce
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        loop.set_postfix(loss=running_loss / (loop.n + 1))

    print(f"→ Epoch {epoch + 1} avg loss: {running_loss / len(loader):.4f}")

# — 6. 保存微调后的 student 模型 —
student.save_pretrained(OUTPUT_DIR)
tokenizer_s.save_pretrained(OUTPUT_DIR)
print(f"✅ Distilled {student_model_name} saved to {OUTPUT_DIR}")


In [1]:
import time

import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from scipy.special import softmax
from tqdm import tqdm

# 1. 模型路径
# model_name = 'distilroberta-base-sst2-distilled'
model_name = 'distill_oracle1_TinyBERT-4L-312D-SST-2_epoch20'
proxy_model_dir_pre = '/home/wangshuo/resource/AIModels/Finetune/TE/sst2/'
model_dir_pre = '/home/wangshuo/resource/AIModels/NLP/TE/'
MODEL_PATH = proxy_model_dir_pre + model_name
# MODEL_PATH = model_dir_pre + model_name

# 2. 手动加载 tokenizer 和 model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
model.half()
model.to(device).eval()

# 3. 手动写标签映射（sst2 两分类）
id2label = {
    0: "negative",
    1: "positive"
}


# 4. （可选）预处理函数
def preprocess(text: str) -> str:
    # 这里 SST-2 不需要特殊替换，简单 strip 即可
    return text.strip()


# 5. 读取 CSV
datadir = "/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5"
in_csv = f"{datadir}/comment.csv"
out_csv = f"{datadir}/comment.csv"
df = pd.read_csv(in_csv)
texts = df['body'].fillna("").astype(str).tolist()


# 6. 批量推理函数
def infer_batch(batch_texts):
    # 6.1 预处理 + 分词
    inputs = tokenizer(
        [preprocess(t) for t in batch_texts],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    # 6.2 模型推理
    with torch.no_grad():
        logits = model(**inputs).logits.cpu().numpy()

    probs = softmax(logits, axis=1)
    # 返回每条文本的 positive 类概率（索引 1）
    return probs[:, 1]


# 7. 遍历所有文本
batch_size = 64
all_scores = []
batch_times = []
for i in tqdm(range(0, len(texts), batch_size), desc=f"{model_name} 推理"):
    batch = texts[i: i + batch_size]
    t0 = time.perf_counter()
    scs = infer_batch(batch)
    t1 = time.perf_counter()
    all_scores.extend(scs.tolist())
    batch_times.append(t1 - t0)

min_time = min(batch_times)
max_time = max(batch_times)
avg_time = sum(batch_times) / len(batch_times)

# 吞吐量 = batch_size / 时间
max_throughput = 1 / min_time  # 最快时的最大吞吐
min_throughput = 1 / max_time  # 最慢时的最小吞吐
avg_throughput = 1 / avg_time  # 平均吞吐

print(f"每个 batch 推理耗时 (s)：最短={min_time:.4f}, 最长={max_time:.4f}, 平均={avg_time:.4f}")
print(f"吞吐量 (batch/s)：最大={max_throughput:.2f}, 最小={min_throughput:.2f}, 平均={avg_throughput:.2f}")

# 8. 写回 CSV
model_label = 'proxy3dd1'
df[f'ML2_{model_label}_probability'] = all_scores
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"情感分析结果已保存到：{out_csv}")


distill_oracle1_TinyBERT-4L-312D-SST-2_epoch20 推理: 100%|██████████| 2052/2052 [01:06<00:00, 30.82it/s]


每个 batch 推理耗时 (s)：最短=0.0083, 最长=0.4807, 平均=0.0321
吞吐量 (batch/s)：最大=120.69, 最小=2.08, 平均=31.16
情感分析结果已保存到：/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/comment.csv
